# Loading and preparing data 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import scirpy as ir
import scanpy as sc
import anndata as ad

import pandas as pd
import numpy as np
import awkward as ak

import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import sys
import warnings

In [ ]:
ir.__version__

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

## Custom modules

In [ ]:
sys.path.append('bin/')
from adata_qc import *

## Settup rpy2 

In [ ]:
os.environ['R_HOME']='/nobackup/peer/fdeckert/miniconda3/envs/r.4.4.1-FD20250213HSCT/lib/R'

In [ ]:
import rpy2
%load_ext rpy2.ipython

## Figures 

In [ ]:
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, facecolor='white')
mpl.rcParams['figure.facecolor']='white'

## Convert Airr VDJDB to TCRdist

In [ ]:
# Load IEDB via Scirpy
adata_vdjdb=ir.datasets.vdjdb()

In [ ]:
# Filter by TCR
ir.pp.index_chains(adata_vdjdb)
ir.tl.chain_qc(adata_vdjdb)

adata_vdjdb = adata_vdjdb[
    (adata_vdjdb.obs["receptor_type"] == "TCR") & 
    (adata_vdjdb.obs["receptor_subtype"] == "TRA+TRB") & 
    (adata_vdjdb.obs["chain_pairing"] == "single pair")
].copy()

# Convert airr to TCRdist format
airr_df = ak.to_dataframe(adata_vdjdb.obsm["airr"]).reset_index()

tra = (
    airr_df.loc[airr_df["locus"] == "TRA", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_a_gene",
        "j_call": "j_a_gene",
        "junction_aa": "cdr3_a_aa",
    })
)

trb = (
    airr_df.loc[airr_df["locus"] == "TRB", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_b_gene",
        "j_call": "j_b_gene",
        "junction_aa": "cdr3_b_aa",
    })
)

cell_vdjdb_df = (
    tra.merge(trb, on="entry", how="outer")
       .sort_values("entry")
       .reset_index(drop=True)
)

cell_vdjdb_df = cell_vdjdb_df[["v_a_gene", "j_a_gene", "cdr3_a_aa", "v_b_gene", "j_b_gene", "cdr3_b_aa"]]

# Prepare meta data
adata_vdjdb.obs['antigen.species'] = adata_vdjdb.obs['antigen.species'].cat.add_categories(["Unknown"]).fillna("Unknown")
adata_vdjdb.obs['antigen.gene'] = adata_vdjdb.obs['antigen.gene'].cat.add_categories(["Unknown"]).fillna("Unknown")
adata_vdjdb.obs['antigen.epitope'] = adata_vdjdb.obs['antigen.epitope'].cat.add_categories(["Unknown"]).fillna("Unknown")

obs = adata_vdjdb.obs[['antigen.species', 'antigen.gene', 'antigen.epitope']].reset_index(drop=True)
obs.columns = ['antigen_species_vdjdb', 'antigen_gene_vdjdb', 'antigen_epitope_vdjdb']

# Join tcr with meta data 
cell_vdjdb_df = pd.concat([cell_vdjdb_df, obs], axis=1)

# Drop duplicates
cell_vdjdb_df = cell_vdjdb_df.dropna().drop_duplicates()

# Save result
cell_vdjdb_df.to_csv('result/ir/tcrdist/cell_vdjdb_df.csv')

## Convert Airr IEDB to TCRdist

In [ ]:
# Load IEDB via Scirpy
adata_iedb=ir.datasets.iedb()

In [ ]:
# Filter by TCR
ir.pp.index_chains(adata_iedb)
ir.tl.chain_qc(adata_iedb)

adata_iedb = adata_iedb[
    (adata_iedb.obs["receptor_type"] == "TCR") & 
    (adata_iedb.obs["receptor_subtype"] == "TRA+TRB") & 
    (adata_iedb.obs["chain_pairing"] == "single pair")
].copy()

# Convert airr to TCRdist format
airr_df = ak.to_dataframe(adata_iedb.obsm["airr"]).reset_index()

tra = (
    airr_df.loc[airr_df["locus"] == "TRA", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_a_gene",
        "j_call": "j_a_gene",
        "junction_aa": "cdr3_a_aa",
    })
)

trb = (
    airr_df.loc[airr_df["locus"] == "TRB", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_b_gene",
        "j_call": "j_b_gene",
        "junction_aa": "cdr3_b_aa",
    })
)

cell_iedb_df = (
    tra.merge(trb, on="entry", how="outer")
       .sort_values("entry")
       .reset_index(drop=True)
)

cell_iedb_df = cell_iedb_df[["v_a_gene", "j_a_gene", "cdr3_a_aa", "v_b_gene", "j_b_gene", "cdr3_b_aa"]]

# Prepare meta data
obs = adata_iedb.obs[['Epitope Source Organism', 'Epitope Source Molecule', 'Epitope IEDB IRI']].reset_index(drop=True)
obs.columns = ['epitope_organism_iedb', 'epitope_molecule_iedb', 'epitope_iri_iedb']

# Join tcr with meta data 
cell_iedb_df = pd.concat([cell_iedb_df, obs], axis=1)

# Filter out old gene names that don't map to TCRdist data base
for vj_col in ["v_a_gene", "j_a_gene", "v_b_gene", "j_b_gene"]:
    cell_iedb_df = cell_iedb_df[~cell_iedb_df[vj_col].astype(str).str.contains("/")]

# Drop duplicates
cell_iedb_df = cell_iedb_df.dropna().drop_duplicates()

# # Save result
cell_iedb_df.to_csv('result/ir/tcrdist/cell_iedb_df.csv')

# Import AnnData

In [ ]:
adata=sc.read_h5ad('data/object/qc.h5ad')

## Sample path of 10x TCR data 

In [ ]:
adata.obs['sample_path_tcr']=np.where(adata.obs['sample_path'].str.contains('S_'), adata.obs['sample_path'].astype(str).replace('_transcriptome', '_VDJ', regex=True), adata.obs['sample_path'].astype(str).replace('_transcriptome', '_abTCR_VDJ', regex=True))

In [ ]:
# df_2 = adata.obs[['sample_path', 'sample_path_tcr']].drop_duplicates()
# df_1 = pd.read_csv('meta.csv')
# df_1.merge(df_2, on="sample_path", how="left").to_csv('meta_tmp.csv')

## Cell type annotation 

In [ ]:
annotation_file = 'result/annotation/annotation.csv'
annotation = pd.read_csv(annotation_file, index_col=0)

In [ ]:
adata.obs = adata.obs.merge(annotation, left_index=True, right_index=True, how='left')

## Genotype class

In [ ]:
genotype_file = 'result/genotype/genotype.csv'
genotype = pd.read_csv(genotype_file, index_col=0)

In [ ]:
adata.obs = adata.obs.merge(genotype, left_index=True, right_index=True, how='left')

## Filter AnnData 

In [ ]:
adata = adata[adata.obs.celltype_high!='$Artefact$']
adata = adata[adata.obs["celltype_high"] == '$T\ cell$']
adata = adata[~adata.obs.celltype_high.isna()]
adata = adata[adata.obs.genotype_class.isin(['Donor', 'Host'])]

In [ ]:
adata = adata.copy()

# Imort TCR 

In [ ]:
adata_tcr_dict=[]
for sample_path_tcr in adata.obs['sample_path_tcr'].unique(): 
    
    print(sample_path_tcr)
    
    # Import TCR 10x data
    adata_tcr=ir.io.read_10x_vdj(sample_path_tcr+'/filtered_contig_annotations.csv')
    adata_cnt=adata[adata.obs['sample_path_tcr']==sample_path_tcr]
    
    # Get cell_id index from meta data and add to adata_tcr
    barcode_idx='_'+adata_cnt.obs_names[1].split('_')[1]
    adata_tcr.obs_names=adata_tcr.obs_names+barcode_idx

    # Subset for obs names union 
    obs_names_union=set(adata_tcr.obs_names) & set(adata_cnt.obs_names)
    adata_tcr=adata_tcr[adata_tcr.obs_names.isin(obs_names_union)]
    adata_cnt=adata_cnt[adata_cnt.obs_names.isin(obs_names_union)]
    
    # Add obs data to adata tcr
    adata_cnt.obs=adata_cnt.obs[list(set(adata_cnt.obs.columns) - set(adata_tcr.obs.columns))]
    adata_tcr.obs=adata_cnt.obs.join(adata_tcr.obs)
    
    # print(adata.obsm['X_umap'])
    adata_tcr.obsm['X_umap']=adata_cnt.obsm['X_umap']
    
    # Output to dict
    adata_tcr_dict.append(adata_tcr)

In [ ]:
adata_tcr=ad.concat(adata_tcr_dict)

# TCR QC 

In [ ]:
ir.pp.index_chains(adata_tcr)
ir.tl.chain_qc(adata_tcr)

In [ ]:
ax=ir.pl.group_abundance(adata_tcr, groupby='chain_pairing', target_col='patient_id')

In [ ]:
ax=ir.pl.group_abundance(adata_tcr, groupby='receptor_subtype', target_col='patient_id')

## Convert Airr to TCRdist

In [ ]:
# Filter by TCR 
adata_tcr = adata_tcr[

    (adata_tcr.obs["receptor_type"] == "TCR") & 
    (adata_tcr.obs["receptor_subtype"] == "TRA+TRB") & 
    (adata_tcr.obs["chain_pairing"] == "single pair")

].copy()

# Convert airr to TCRdist format
airr_df = ak.to_dataframe(adata_tcr.obsm["airr"]).reset_index()

tra = (
    airr_df.loc[airr_df["locus"] == "TRA", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_a_gene",
        "j_call": "j_a_gene",
        "junction_aa": "cdr3_a_aa",
    })
)

trb = (
    airr_df.loc[airr_df["locus"] == "TRB", ["entry", "v_call", "j_call", "junction_aa"]]
    .rename(columns={
        "v_call": "v_b_gene",
        "j_call": "j_b_gene",
        "junction_aa": "cdr3_b_aa",
    })
)

cell_df = (
    tra.merge(trb, on="entry", how="outer")
       .sort_values("entry")
       .reset_index(drop=True)
)

cell_df = cell_df[["v_a_gene", "j_a_gene", "cdr3_a_aa", "v_b_gene", "j_b_gene", "cdr3_b_aa"]]

# Prepare meta data
obs = adata_tcr.obs[['cell_id', 'patient_id', 'time_point', 'hto_demux_class', 'genotype_class']].rename(columns={'cell_id': 'barcode'}).reset_index(drop=True)

# Join tcr with meta data 
cell_df = pd.concat([cell_df, obs], axis=1)

# Correct allele names 
for vj_col in ["v_a_gene", "j_a_gene", "v_b_gene", "j_b_gene"]:
    cell_df[vj_col] = cell_df[vj_col].where(cell_df[vj_col].str.contains(r"\*"), cell_df[vj_col] + "*01") # Set allele https://tcrdist3.readthedocs.io/en/latest/inputs.html

## Set clone ID

In [ ]:
# Clone cols 
clone_cols = ['v_a_gene','j_a_gene','cdr3_a_aa', 'v_b_gene','j_b_gene','cdr3_b_aa']

# Make clone ids base on TCR information 
clone_counts = cell_df.groupby(clone_cols).size().rename('n_cells').reset_index()
clone_counts = clone_counts.sort_values(['n_cells'] + clone_cols, ascending=[False] + [True]*len(clone_cols))

# Rank clone size and make id
clone_counts['clone_rank'] = clone_counts.reset_index(drop=True).index
clone_counts['clone_id'] = 'clone_id_' + clone_counts['clone_rank'].astype(str)

# Map back to cell_df
cell_df = cell_df.merge(clone_counts[clone_cols + ['clone_id']], on=clone_cols, how='left')

## Save cell df for TCRdist analysis

In [ ]:
# Save result
cell_df.to_csv('result/ir/tcrdist/cell_df.csv')

# Compute clono types

In [ ]:
# Dist based on AA similarity 
ir.pp.ir_dist(adata_tcr, metric='tcrdist', sequence='aa')
ir.tl.define_clonotypes(adata_tcr, receptor_arms='all', dual_ir='any')

In [ ]:
ir.tl.clonotype_network(adata_tcr, min_cells=5)
ir.pl.clonotype_network(adata_tcr, color='time_point', show_labels=False, panel_size=(10, 10), base_size=20, edges=True)

# Clonotype analysis

In [ ]:
ir.tl.clonal_expansion(adata_tcr, clip_at=2)

# Match VDJDB

In [ ]:
ir.pp.ir_dist(adata_tcr, adata_vdjdb, metric='tcrdist', sequence='aa')
ir.tl.ir_query(adata_tcr, adata_vdjdb, metric='tcrdist', sequence='aa', receptor_arms='any', dual_ir='any')
ir.tl.ir_query_annotate(adata_tcr, adata_vdjdb, metric='tcrdist', sequence='aa', include_ref_cols=['antigen.species', 'antigen.gene'], strategy='most-frequent',)
adata_tcr.obs.rename(columns={'antigen.species':'species_vdjdb', 'antigen.gene':'antigen_vdjdb'}, inplace=True)

# Match IEDB

In [ ]:
ir.pp.ir_dist(adata_tcr, adata_iedb, metric='tcrdist', sequence='aa')
ir.tl.ir_query(adata_tcr, adata_iedb, metric='tcrdist', sequence='aa', receptor_arms='any', dual_ir='any')
ir.tl.ir_query_annotate(adata_tcr, adata_iedb, metric='tcrdist', sequence='aa', include_ref_cols=['Epitope Source Molecule', 'Epitope Source Organism'], strategy='most-frequent',)
adata_tcr.obs.rename(columns={'Epitope Source Organism':'species_iedb', 'Epitope Source Molecule':'antigen_iedb'}, inplace=True)

# Get results 

In [ ]:
# Get IR from obs
ir_obs = adata_tcr.obs[['receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'species_vdjdb', 'antigen_vdjdb', 'antigen_iedb', 'species_iedb']]

In [ ]:
# Get alpha TCR information 
df_vj = pd.DataFrame({
    "v_call_vj": ir.get.airr(adata_tcr, "v_call", chain="VJ_1"),
    "d_call_vj": ir.get.airr(adata_tcr, "d_call", chain="VJ_1"),
    "j_call_vj": ir.get.airr(adata_tcr, "j_call", chain="VJ_1"),
    "c_call_vj": ir.get.airr(adata_tcr, "c_call", chain="VJ_1"),
    "cdr3_aa_vj": ir.get.airr(adata_tcr, "cdr3_aa", chain="VJ_1")
}, index=adata_tcr.obs_names)

In [ ]:
# Get beta TCR information 
df_vdj = pd.DataFrame({
    "v_call_vdj": ir.get.airr(adata_tcr, "v_call", chain="VDJ_1"),
    "d_call_vdj": ir.get.airr(adata_tcr, "d_call", chain="VDJ_1"),
    "j_call_vdj": ir.get.airr(adata_tcr, "j_call", chain="VDJ_1"),
    "c_call_vdj": ir.get.airr(adata_tcr, "c_call", chain="VDJ_1"),
    "cdr3_aa_vdj": ir.get.airr(adata_tcr, "cdr3_aa", chain="VDJ_1")
}, index=adata_tcr.obs_names)

# Save results 

In [ ]:
pd.concat([ir_obs, df_vj, df_vdj], axis=1, join="inner").to_csv('result/ir/scirpy/scirpy.csv')